##### Pre-processing data

In [51]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score

In [52]:
matches = pd.read_csv("combined_3seasons.csv", index_col = False)
matches.head()

,date,time,home_team,away_team,score,match_url,Expected_goals_(xG)_home,Expected_goals_(xG)_away,Ball_possession_home,Ball_possession_away,...,Errors_leading_to_shot_home,Errors_leading_to_shot_away,Errors_leading_to_goal_home,Errors_leading_to_goal_away,xGOT_faced_home,xGOT_faced_away,Goals_prevented_home,Goals_prevented_away,Headed_goals_home,Headed_goals_away
0,30.04.2023,21:00,Cremonese,Verona,1:1,https://www.flashscore.co.uk/match/football/AP...,0.49,1.21,51%,49%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,08.01.2023,19:30,Salernitana,Torino,1:1,https://www.flashscore.co.uk/match/football/pl...,NaN,NaN,39%,61%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,28.01.2023,22:00,Empoli,Torino,2:2,https://www.flashscore.co.uk/match/football/tE...,NaN,NaN,47%,53%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04.01.2023,23:30,AS Roma,Bologna,1:0,https://www.flashscore.co.uk/match/football/bZ...,NaN,NaN,39%,61%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05.06.2023,03:00,Lecce,Bologna,2:3,https://www.flashscore.co.uk/match/football/Kj...,1.64,0.87,39%,61%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Cleaning the data

In [53]:
# Since home_team and away_team columns have the same domain of values, 
# we should encode them into categorical value using the same mapping
categories = sorted(matches['home_team'].unique())
matches['date'] = pd.to_datetime(matches['date'], format='%d.%m.%Y')
matches['home_team_code'] = pd.Categorical(matches['home_team'], categories = categories).codes
matches['away_team_code'] = pd.Categorical(matches['away_team'], categories = categories).codes
matches['hour'] = matches['time'].str.replace(":.+", "", regex = True).astype('int')
matches['day_code'] = matches['date'].dt.dayofweek


matches['Passes_home'] = matches['Passes_home'].str.split(',').str[0].str.replace('%','').astype('float')/100
matches['Passes_away'] = matches['Passes_away'].str.split(',').str[0].str.replace('%','').astype('float')/100
matches['Ball_possession_home'] = matches['Ball_possession_home'].str.replace('%','').astype('float')/100

matches['score'] = matches['score'].astype(str)
matches['result'] = matches.apply(lambda row: 
    1 if int(row['score'].split(':')[0]) > int(row['score'].split(':')[1]) else -1,
    axis = 1
)

# 1 if home team wins, -1 if away team wins, 0 if draw
matches.shape





(1140, 81)

In [54]:
matches = matches.sort_values(by = 'date').reset_index(drop = True)


### Handling the missing data

In [55]:
# I wanna find what columns contain NaN values
columns_with_nan = matches.columns[matches.isna().any()].tolist()


In [56]:
nan_counts = matches.isna().sum()
nan_percentage = matches.isna().mean() * 100
nan_summary = pd.DataFrame({'Nan_Count' : nan_counts, 'Nan_Percentage' : nan_percentage})
print('Nan Summary:\n', nan_summary[nan_summary['Nan_Count'] > 0])
nan_summary.dtypes

Nan Summary:
                           Nan_Count  Nan_Percentage
Expected_goals_(xG)_home        216       18.947368
Expected_goals_(xG)_away        216       18.947368
Ball_possession_home              1        0.087719
Ball_possession_away              1        0.087719
Total_shots_home                  1        0.087719
...                             ...             ...
xGOT_faced_away                1050       92.105263
Goals_prevented_home           1050       92.105263
Goals_prevented_away           1050       92.105263
Headed_goals_home              1052       92.280702
Headed_goals_away              1052       92.280702

[70 rows x 2 columns]


Nan_Count           int64
Nan_Percentage    float64
dtype: object

In [57]:
rows_with_nan = matches[matches.isna().any(axis=1)]
print(f'rows with >=1 NaN value: {len(rows_with_nan)} out of {len(matches)}')

rows with >=1 NaN value: 1133 out of 1140


In [58]:
# Let's find columns with NaN percentage < 20% 
threshold = 15
columns_to_keep = nan_summary[nan_summary['Nan_Percentage'] <= threshold].index.tolist()
if 'result' not in columns_to_keep:
    columns_to_keep.append('result')
# columns_to_keep
matches = matches[columns_to_keep]

In [59]:
columns_to_stay = ['date','home_team', 'away_team',
              'home_team_code', 'away_team_code', 'hour', 'day_code', 
              'Ball_possession_home', 
              'Total_shots_home', 'Total_shots_away',
              'Goalkeeper_saves_home', 'Goalkeeper_saves_away', 
              'Corner_kicks_home', 'Corner_kicks_away',
              'Passes_home', 'Passes_away',
              'Free_kicks_home', 'Free_kicks_away',
              'result']
predictors = ['date',
              'home_team_code', 'away_team_code', 'hour', 'day_code', 
              'Ball_possession_home', 
              'Total_shots_home', 'Total_shots_away',
              'Goalkeeper_saves_home', 'Goalkeeper_saves_away', 
              'Corner_kicks_home', 'Corner_kicks_away',
              'Passes_home', 'Passes_away',
              'Free_kicks_home', 'Free_kicks_away']

predictors_2 = ['date',
              'home_team_code', 'away_team_code', 'hour', 'day_code', 
              'Ball_possession_home', 
              'Total_shots_home', 'Total_shots_away',
              'Goalkeeper_saves_home', 'Goalkeeper_saves_away', 
              'Corner_kicks_home', 'Corner_kicks_away',
              'Passes_home', 'Passes_away',
              'Free_kicks_home', 'Free_kicks_away']
# target = ['result']
matches = matches[columns_to_stay]
matches.dtypes

date                     datetime64[ns]
home_team                        object
away_team                        object
home_team_code                     int8
away_team_code                     int8
hour                              int64
day_code                          int32
Ball_possession_home            float64
Total_shots_home                float64
Total_shots_away                float64
Goalkeeper_saves_home           float64
Goalkeeper_saves_away           float64
Corner_kicks_home               float64
Corner_kicks_away               float64
Passes_home                     float64
Passes_away                     float64
Free_kicks_home                 float64
Free_kicks_away                 float64
result                            int64
dtype: object

In [60]:
# Handle remaining NaN values for Numeric columns
numeric_cols = matches.select_dtypes(include = ['float64','int64']).columns
matches[numeric_cols] = matches[numeric_cols].fillna(matches[numeric_cols].median())

In [61]:
# Handle remaining NaN values for Categorical columns
categorical_cols = matches.select_dtypes(include = ['int8','int32']).columns
matches[categorical_cols] = matches[categorical_cols].fillna(matches[categorical_cols].mode().iloc[0])

In [62]:
# Most of the rows are still present in the table now
matches.shape

(1140, 19)

### ML Model Fine-tuning 

#### Improving precision with rolling averages(skipped for now)

In [ ]:
from rolling_tuning import compute_rolling_features
for window in [3, 5, 10, 15]:
    temp_matches = compute_rolling_features(matches, window, predictors, 'result')
    temp_matches.dropna(inplace = True)
    
    predictors = ['home_team_code', 'away_team_code', 'hour', 'day_code', 
              'Ball_possession_home', 
              'Total_shots_home', 'Total_shots_away',
              'Goalkeeper_saves_home', 'Goalkeeper_saves_away', 
              'Corner_kicks_home', 'Corner_kicks_away',
              'Passes_home', 'Passes_away',
              'Free_kicks_home', 'Free_kicks_away']
    home_stats = ['Ball_possession_home','Total_shots_home','Goalkeeper_saves_home','Corner_kicks_home','Passes_home','Free_kicks_home']
    away_stats = ['Total_shots_away','Goalkeeper_saves_away','Corner_kicks_away','Passes_away','Free_kicks_away']


In [65]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score
# --- Step 2: Train + Evaluate RF with rolling tuning ---
def train_and_evaluate_rf_with_rolling(matches, base_predictors, home_stats, away_stats, split_date='2024-10-05', windows=[2, 3, 5, 7]):
    results = []

    for window in windows:
        data = compute_rolling_features(matches.copy(), home_stats, away_stats, window)
        data.dropna(inplace=True)

        rolling_features = [col for col in data.columns if 'rolling' in col]
        predictors = base_predictors + rolling_features

        train = data[data['date'] < split_date]
        test = data[data['date'] >= split_date]

        rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)
        rf.fit(train[predictors], train['result'])
        preds = rf.predict(test[predictors])

        acc = accuracy_score(test['result'], preds)
        precision = precision_score(test['result'], preds, average='macro')

        results.append({
            'window_size': window,
            'accuracy': acc,
            'precision': precision
        })

    return pd.DataFrame(results)


# --- Step 3: Define columns ---
base_predictors = [
    'home_team_code', 'away_team_code', 'hour', 'day_code', 
    'Ball_possession_home', 'Total_shots_home', 'Total_shots_away',
    'Goalkeeper_saves_home', 'Goalkeeper_saves_away',
    'Corner_kicks_home', 'Corner_kicks_away',
    'Passes_home', 'Passes_away',
    'Free_kicks_home', 'Free_kicks_away'
]

home_stats = [
    'Ball_possession_home', 'Total_shots_home', 'Goalkeeper_saves_home',
    'Corner_kicks_home', 'Passes_home', 'Free_kicks_home'
]

away_stats = [
    'Total_shots_away', 'Goalkeeper_saves_away',
    'Corner_kicks_away', 'Passes_away', 'Free_kicks_away'
]

# --- Step 4: Run and print results ---
results_df = train_and_evaluate_rf_with_rolling(
    matches, 
    base_predictors, 
    home_stats, 
    away_stats,
    split_date='2024-10-05', 
    windows=[3, 5, 10, 15]
)

print(results_df)

   window_size  accuracy  precision
0            3  0.643750   0.625422
1            5  0.681250   0.668430
2           10  0.696875   0.696300
3           15  0.706250   0.699883


#### Fine-tuning ForrestClassifier params and Rolling window size 

In [73]:
base_predictors = [
    'home_team_code', 'away_team_code', 'hour', 'day_code', 
    'Ball_possession_home', 'Total_shots_home', 'Total_shots_away',
    'Goalkeeper_saves_home', 'Goalkeeper_saves_away',
    'Corner_kicks_home', 'Corner_kicks_away',
    'Passes_home', 'Passes_away',
    'Free_kicks_home', 'Free_kicks_away'
]

home_stats = [
    'Ball_possession_home', 'Total_shots_home', 'Goalkeeper_saves_home',
    'Corner_kicks_home', 'Passes_home', 'Free_kicks_home'
]

away_stats = [
    'Total_shots_away', 'Goalkeeper_saves_away',
    'Corner_kicks_away', 'Passes_away', 'Free_kicks_away'
]

import pandas as pd
import itertools
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score

# --- Rolling Feature Generator ---
def compute_rolling_features(df, home_stats, away_stats, window):
    df = df.sort_values("date").reset_index(drop=True)

    def calc_rolling(df, team_col, stat_cols, prefix):
        all_teams = []
        for team_id in df[team_col].unique():
            team_df = df[df[team_col] == team_id].copy()
            team_df = team_df.sort_values("date")
            for stat in stat_cols:
                colname = f"{prefix}_{stat}_rolling_{window}"
                team_df[colname] = team_df[stat].rolling(window=window, min_periods=1).mean().shift(1)
            all_teams.append(team_df)
        return pd.concat(all_teams)

    df = calc_rolling(df, "home_team_code", home_stats, "home")
    df = calc_rolling(df, "away_team_code", away_stats, "away")
    return df

# --- Model Tuning over Rolling Window and RF Parameters ---
def tune_rf_with_rolling_grid(
    matches,
    base_predictors,
    home_stats,
    away_stats,
    split_date='2024-10-05',
    window_sizes=[2, 3, 5],
    n_estimators_list=[50, 100],
    min_samples_split_list =[5, 10, 20, 50],
    max_depth_list=[5, 10, None]
):
    results = []

    # Grid search across all combinations
    for window, n_estimators, max_depth, min_samples_split in itertools.product(window_sizes, n_estimators_list, max_depth_list, min_samples_split_list):
        data = compute_rolling_features(matches.copy(), home_stats, away_stats, window)
        data.dropna(inplace=True)

        rolling_features = [col for col in data.columns if 'rolling' in col]
        predictors = base_predictors + rolling_features

        train = data[data['date'] < split_date]
        test = data[data['date'] >= split_date]

        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=1
        )

        rf.fit(train[predictors], train['result'])
        preds = rf.predict(test[predictors])

        acc = accuracy_score(test['result'], preds)
        precision = precision_score(test['result'], preds, average='macro')
        f1 = f1_score(test['result'], preds, average='macro')
        
        results.append({
            'window_size': window,
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'min_samples_split': min_samples_split,
            'accuracy': acc,
            'precision': precision,
            'f1_score': f1
        })

    return pd.DataFrame(results)

results_df = tune_rf_with_rolling_grid(
    matches,
    base_predictors,
    home_stats,
    away_stats,
    window_sizes=[2, 3, 5, 7],
    n_estimators_list=[30 ,40,50, 100, 150],
    min_samples_split_list = [5,10, 20, 50],
    max_depth_list=[5, 10, None]
)

print(results_df.sort_values("accuracy", ascending=False))

     window_size  n_estimators  max_depth  min_samples_split  accuracy  \
198            7            40       10.0                 20  0.709375   
186            7            30       10.0                 20  0.706250   
238            7           150        NaN                 20  0.703125   
209            7            50       10.0                 10  0.703125   
120            5            30        5.0                  5  0.700000   
..           ...           ...        ...                ...       ...   
0              2            30        5.0                  5  0.634375   
5              2            30       10.0                 10  0.631250   
41             2           100       10.0                 10  0.631250   
13             2            40        5.0                 10  0.631250   
9              2            30        NaN                 10  0.625000   

     precision  f1_score  
198   0.699392  0.687313  
186   0.696124  0.683382  
238   0.695685  0.674431  
209

### Creating ML Model (after fine-tuning)
window_size = 7, n_estimators = 40, max_depth = 10, min_samples_list = 20

In [94]:
data = compute_rolling_features(matches.copy(), home_stats, away_stats, window = 7)
data.dropna(inplace=True)

rolling_features = [col for col in data.columns if 'rolling' in col]
predictors = base_predictors + rolling_features

train = data[data['date']< '2024-10-05']
test = data[data['date']>= '2024-10-05']
data.head(10)
# data.columns


,date,home_team,away_team,home_team_code,away_team_code,hour,day_code,Ball_possession_home,Total_shots_home,Total_shots_away,...,home_Total_shots_home_rolling_7,home_Goalkeeper_saves_home_rolling_7,home_Corner_kicks_home_rolling_7,home_Passes_home_rolling_7,home_Free_kicks_home_rolling_7,away_Total_shots_away_rolling_7,away_Goalkeeper_saves_away_rolling_7,away_Corner_kicks_away_rolling_7,away_Passes_away_rolling_7,away_Free_kicks_away_rolling_7
39,2022-09-02,Atalanta,Torino,2,22,2,4,0.41,12.0,8.0,...,11.000000,6.000000,4.000000,0.790000,9.000000,13.500000,4.000000,5.000000,0.745000,10.000000
53,2022-09-11,Inter,Torino,11,22,0,6,0.57,16.0,9.0,...,18.500000,1.500000,8.000000,0.880000,12.000000,11.666667,3.333333,4.000000,0.766667,15.000000
70,2022-10-01,Napoli,Torino,16,22,21,5,0.49,15.0,16.0,...,23.000000,1.666667,8.666667,0.890000,21.666667,11.000000,3.000000,4.250000,0.780000,14.750000
105,2022-10-23,Udinese,Torino,23,22,18,6,0.54,16.0,16.0,...,12.800000,3.200000,4.800000,0.794000,14.400000,12.000000,3.200000,4.200000,0.786000,12.600000
127,2022-11-06,Bologna,Torino,3,22,19,6,0.56,9.0,7.0,...,13.000000,2.500000,5.333333,0.768333,17.833333,12.666667,3.166667,4.166667,0.783333,13.666667
145,2022-11-13,AS Roma,Torino,1,22,22,6,0.48,13.0,10.0,...,16.000000,2.000000,5.833333,0.825000,13.500000,11.857143,3.142857,3.714286,0.778571,13.714286
161,2023-01-08,Salernitana,Torino,18,22,19,6,0.39,11.0,23.0,...,11.714286,3.571429,4.142857,0.788571,14.857143,11.428571,3.142857,2.714286,0.777143,16.000000
184,2023-01-22,Fiorentina,Torino,8,22,3,6,0.62,19.0,9.0,...,17.142857,1.571429,6.571429,0.828571,16.428571,12.714286,2.857143,3.285714,0.795714,15.571429
191,2023-01-28,Empoli,Torino,7,22,22,5,0.47,11.0,16.0,...,13.857143,4.000000,4.142857,0.795714,13.857143,12.857143,3.285714,3.428571,0.785714,13.571429
210,2023-02-11,AC Milan,Torino,0,22,3,5,0.46,9.0,12.0,...,16.000000,3.000000,4.714286,0.797143,13.142857,13.857143,3.285714,3.428571,0.780000,14.571429


In [80]:
# Linear Regression is not suitable for classification problems, so we use Random Forest Classifier 
# bcs Linear Regression Model doesn't understand categorical variables well
rf = RandomForestClassifier(n_estimators = 40, max_depth = 10, min_samples_split = 20, random_state = 1)


In [81]:
train.columns

Index(['date', 'home_team', 'away_team', 'home_team_code', 'away_team_code',
       'hour', 'day_code', 'Ball_possession_home', 'Total_shots_home',
       'Total_shots_away', 'Goalkeeper_saves_home', 'Goalkeeper_saves_away',
       'Corner_kicks_home', 'Corner_kicks_away', 'Passes_home', 'Passes_away',
       'Free_kicks_home', 'Free_kicks_away', 'result',
       'home_Ball_possession_home_rolling_7',
       'home_Total_shots_home_rolling_7',
       'home_Goalkeeper_saves_home_rolling_7',
       'home_Corner_kicks_home_rolling_7', 'home_Passes_home_rolling_7',
       'home_Free_kicks_home_rolling_7', 'away_Total_shots_away_rolling_7',
       'away_Goalkeeper_saves_away_rolling_7',
       'away_Corner_kicks_away_rolling_7', 'away_Passes_away_rolling_7',
       'away_Free_kicks_away_rolling_7'],
      dtype='object')

In [82]:
rf.fit(train[predictors], train['result'])

,n_estimators,40
,criterion,'gini'
,max_depth,10
,min_samples_split,20
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [83]:
preds = rf.predict(test[predictors])

In [84]:
acc = accuracy_score(test['result'], preds)
acc

0.709375

In [85]:
combined = pd.DataFrame(dict(actual = test['result'], prediction = preds))
pd.crosstab(index = combined['actual'], columns = combined['prediction'])


prediction,-1,1
actual,,
-1,156,35
1,58,71


In [86]:
from sklearn.metrics import precision_score
precision_score(test['result'], preds)


0.6698113207547169

In [87]:
grouped_matches = matches.groupby('home_team')
group = grouped_matches.get_group('AC Milan')


### Using the model for predicting live match

In [ ]:
# Function to calculate rolling averages for a team
def get_team_rolling_averages(team, df, stat_cols, n_matches=5):
    """
    Calculate rolling averages for a team over their last n_matches for given stats.
    Combines home and away stats (e.g., home_team's ball_possession_home with away_team's ball_possession_away).
    """
    team_matches = df[(df['home_team'] == team) | (df['away_team'] == team)].sort_values('date').tail(n_matches)
    averages = {}
    for stat in stat_cols:
        if 'home' in stat:
            stat_values = pd.concat([
                team_matches[team_matches['home_team'] == team][stat],
                team_matches[team_matches['away_team'] == team][stat.replace('_home', '_away')]
            ])
        else:
            stat_values = pd.concat([
                team_matches[team_matches['away_team'] == team][stat],
                team_matches[team_matches['home_team'] == team][stat.replace('_away', '_home')]
            ])
        # Use mean of available values, or default if no data
        averages[stat] = stat_values.mean() if len(stat_values) > 0 else 0
    return averages


In [89]:
# Statistics needed for the prediction: 

           
# Example:
match_instance = pd.DataFrame({
    'home_team_code': [0],
    'away_team_code': [20],
    'hour': [20],
    'day_code': [5],  # Saturday
    'Ball_possession_home': [0.95],
    'Total_shots_home': [40],
    'Total_shots_away': [5],
    'Goalkeeper_saves_home': [30],
    'Goalkeeper_saves_away': [2],
    'Corner_kicks_home': [30],
    'Corner_kicks_away': [2],
    'Passes_home': [1],
    'Passes_away': [0.05],
    'Free_kicks_home': [30],
    'Free_kicks_away': [0]
}, columns = predictors)

# If there are any NaN values in the live match instance
match_instance[numeric_cols.intersection(predictors)] = match_instance[numeric_cols.intersection(predictors)].fillna(matches[numeric_cols.intersection(predictors)].median())
match_instance[categorical_cols.intersection(predictors)] = match_instance[categorical_cols.intersection(predictors)].fillna(matches[categorical_cols.intersection(predictors)].mode().iloc[0])

prediction = rf.predict(match_instance)
pr = rf.predict_proba(match_instance)

result_mapping = {1: "Home Win", -1: "Away Win or Draw"}
print("Predicted result:", result_mapping.get(prediction[0], "Unknown"))
print("Prediction probabilities (Home Win, Away Win or Draw):", pr[0])



Predicted result: Away Win or Draw
Prediction probabilities (Home Win, Away Win or Draw): [0.54621083 0.45378917]


In [90]:
import streamlit as st
import joblib



In [92]:
joblib.dump(rf, 'serie_a_predictor.pkl')
joblib.dump(categories, 'team_categories.pkl')

['team_categories.pkl']

##### Saving processed matches_processed.csv file

In [ ]:
matches.head()
matches.to_csv('matches_processed.csv', index=False)